In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [2]:
data_source = "GTEx_cortex_counts_TMMF_All_501_outliers_removed_top_Qval_mods_PC1_ctype_abundance_filtered_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance"

psi_corr_df = pd.read_csv(f"data/corrs/{data_source}_exon_PSI_corr.csv", index_col=0)
expr_corr_df = pd.read_csv(f"data/corrs/{data_source}_gene_expr_corr.csv", index_col=0)

In [3]:
column_order = [
    "CGE Class", "All GABAergic", "All Neuronal", "Upper layer glutamatergic", "Deep layer glutamatergic", 
    "Oligo", "OPC", "Astro", "Micro/PVM", "VLMC", "Endo", "Peri"
]
psi_corr_df = psi_corr_df[["Gene"] +  column_order]
expr_corr_df = expr_corr_df[column_order]

In [4]:
outdir = f"figures/ctype_exons/{data_source}"

In [5]:
# Get coordinates for each exon

intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file.tab.gz"
intron_table = read_intron_file(intron_file)

intron_coords_df = intron_table['intron'].str.split(r"[:\-]", expand=True).iloc[:, :3]
intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base"]
intron_coords_df.index = intron_table.index
intron_coords_df['exon'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
intron_coords_df = intron_coords_df[intron_coords_df['exon'].isin(psi_corr_df.index)] # Subset to exons in PSI corr data
intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)

exon_coords_df = intron_coords_df.groupby("exon").apply(lambda g: pd.Series({
    "chr": g["chr"].iloc[0],
    "exon_start": g.loc[g.index.str.contains("I1"), "intron_last_base"].values[0] + 1,
    "exon_end": g.loc[g.index.str.contains("I2"), "intron_first_base"].values[0] - 1,
}))

/tmp/ipykernel_3314194/3893562153.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exon_coords_df = intron_coords_df.groupby("exon").apply(lambda g: pd.Series({


In [21]:
intron_coords_df[intron_coords_df['exon'] == "ENSG00000138326_ProteinCoding_1"]

,chr,intron_first_base,intron_last_base,exon
ENSG00000138326_ProteinCoding_1_I1,chr10,78037305,78037438,ENSG00000138326_ProteinCoding_1
ENSG00000138326_ProteinCoding_1_I2,chr10,78037442,78037964,ENSG00000138326_ProteinCoding_1
ENSG00000138326_ProteinCoding_1_SE,chr10,78037305,78037964,ENSG00000138326_ProteinCoding_1


In [22]:
intron_coords_df[intron_coords_df['exon'] == "ENSG00000138326_ProteinCoding_2"]

,chr,intron_first_base,intron_last_base,exon
ENSG00000138326_ProteinCoding_2_I1,chr10,78037305,78037438,ENSG00000138326_ProteinCoding_2
ENSG00000138326_ProteinCoding_2_I2,chr10,78037442,78040203,ENSG00000138326_ProteinCoding_2
ENSG00000138326_ProteinCoding_2_SE,chr10,78037305,78040203,ENSG00000138326_ProteinCoding_2


In [25]:
exon_coords_df.loc['ENSG00000138326_ProteinCoding_2']

chr              chr10
exon_start    78037439
exon_end      78037441
Name: ENSG00000138326_ProteinCoding_2, dtype: object

In [6]:
ctypes = expr_corr_df.columns

## Make plots

In [ ]:
min_psi_corr = 0.25
ascending = True 

for ctype in ctypes:
    print(ctype)
    
    def _safe(name):
        import re
        # simple filename sanitizer
        return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

    pdf_path = f"{outdir}/{_safe(ctype)}_ascending{ascending}.pdf"

    os.makedirs(outdir, exist_ok=True)

    if ascending:
        mask_is_max = (psi_corr_df.iloc[:, 1:].min(axis=1) == psi_corr_df[ctype]) # Subset to exons for which working cell type has the highest PSI corr
        mask_min_psi = (psi_corr_df[ctype] < -min_psi_corr)
    else:
        mask_is_max = (psi_corr_df.iloc[:, 1:].max(axis=1) == psi_corr_df[ctype])
        mask_min_psi = (psi_corr_df[ctype] > min_psi_corr)
        
    mask = mask_min_psi & mask_is_max
    psi_sorted_df = psi_corr_df[mask].sort_values(ctype, ascending=ascending)
    expr_sorted_df = expr_corr_df.merge(
        psi_sorted_df['Gene'], left_index=True, right_on="Gene", how="right"
    )
    
    with PdfPages(pdf_path) as pdf:
        for idx, row in psi_sorted_df.iterrows():
            exon = "_".join(idx.split("_")[1:])
            gene = row['Gene']
            
            start_coord = exon_coords_df.loc[idx]['exon_start']
            end_coord = exon_coords_df.loc[idx]['exon_end']
            exon_len = end_coord - start_coord + 1

            fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
            fig.suptitle(f'{ctype}', fontsize=14)
            
            # Bar plot of exon PSI correlation by cell type
            colors = ['salmon' if c == ctype else 'steelblue' for c in psi_corr_df.columns[1:]]
            axes[0].bar(psi_corr_df.columns[1:], psi_corr_df.loc[idx].iloc[1:], color=colors)
            axes[0].set_title(f'{row['Gene']} {exon} PSI correlation\n{exon_len} nts ({start_coord}-{end_coord})')
            axes[0].set_ylabel('Correlation')
            axes[0].axhline(0, color='black', linewidth=0.5)
            axes[0].set_xticks(range(len(psi_corr_df.columns[1:])))
            axes[0].set_xticklabels(psi_corr_df.columns[1:], rotation=45, ha='right', fontsize=8)

            # Bar plot of gene expression correlation by cell type
            axes[1].bar(expr_corr_df.columns, expr_sorted_df.loc[idx].iloc[:-1], color=colors)
            axes[1].set_title(f'Expression Correlation — {row['Gene']}')
            axes[1].set_ylabel('Correlation')
            axes[1].axhline(0, color='black', linewidth=0.5)
            axes[1].set_xticks(range(len(expr_corr_df.columns)))
            axes[1].set_xticklabels(expr_corr_df.columns, rotation=45, ha='right', fontsize=8)

            plt.tight_layout(rect=[0, 0, 1, 1])  # leave room at top for suptitle
            pdf.savefig(fig)
            plt.close(fig)

CGE Class
All GABAergic
All Neuronal
Upper layer glutamatergic
Deep layer glutamatergic
Oligo
OPC
Astro
Micro/PVM
VLMC
Endo
Peri


## Make table of exons per cell type

In [12]:
min_psi_corr = 0.25
ascending = False

ctype_exon_list = []

for ctype in ctypes:
    print(ctype)

    if ascending:
        mask_is_max = (psi_corr_df.iloc[:, 1:].min(axis=1) == psi_corr_df[ctype]) # Subset to exons for which working cell type has the highest PSI corr
        mask_min_psi = (psi_corr_df[ctype] < -min_psi_corr)
    else:
        mask_is_max = (psi_corr_df.iloc[:, 1:].max(axis=1) == psi_corr_df[ctype])
        mask_min_psi = (psi_corr_df[ctype] > min_psi_corr)

    mask = mask_min_psi & mask_is_max    
    psi_sorted_df = psi_corr_df[mask].sort_values(ctype, ascending=ascending)
    exon_coords_df_sorted = exon_coords_df.reindex(psi_sorted_df.index)
    expr_sorted_df = expr_corr_df.merge(
        psi_sorted_df['Gene'], left_index=True, right_on="Gene", how="right"
    )

    exons = psi_sorted_df.index.str.split("_").str[1:].str.join("_")
    start_coords = exon_coords_df_sorted['exon_start']
    end_coords = exon_coords_df_sorted['exon_end']
    exon_lens = end_coords - start_coords + 1
    exon_coords = exon_coords_df_sorted['chr'] + ":" + exon_coords_df_sorted['exon_start'].astype("str") + "-" + exon_coords_df_sorted['exon_end'].astype("str")

    # genes = psi_sorted_df['Gene']

    ctype_exon_list.append(
        pd.DataFrame({
            "Cell_type": ctype,
            "Gene": psi_sorted_df['Gene'].values,
            "Exon": exons.values,
            "PSI_corr": psi_sorted_df[ctype].values,
            "Expr_corr": expr_sorted_df[ctype].loc[psi_sorted_df.index].values,
            "Exon_length": exon_lens.values,
            "Exon_coords": exon_coords.values
            
        }, index=psi_sorted_df.index)
    )
    
ctype_exon_df = pd.concat(ctype_exon_list)

ctype_exon_df.to_csv(f"data/ctype_exons/{data_source}_ascending{ascending}.csv")

CGE Class
All GABAergic
All Neuronal
Upper layer glutamatergic
Deep layer glutamatergic
Oligo
OPC
Astro
Micro/PVM
VLMC
Endo
Peri
